In [2]:
# Environment Setup
!git clone https://github.com/anikanb-32/musicandmemory.git
%cd musicandmemory
!cd /content/musicandmemory && git pull
!pip install -r requirements.txt

Cloning into 'musicandmemory'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 169 (delta 91), reused 56 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 85.25 KiB | 10.66 MiB/s, done.
Resolving deltas: 100% (91/91), done.
/content/musicandmemory
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.8/409.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing insta

In [3]:
# Connect to Google Drive and OpenAI API
from google.colab import userdata, drive
import os, json, time
import pandas as pd
import numpy as np

drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/GenAI Final Project Music&Memory/'

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
client = OpenAI()

Mounted at /content/drive


In [4]:
# Test to see if imported files are empty
!cat /content/musicandmemory/src/retrieval.py
!cat /content/musicandmemory/src/generation.py
!cat /content/musicandmemory/configs/prompts.py

import faiss
import numpy as np
import pandas as pd
from openai import OpenAI

client = OpenAI()

def load_retrieval_system(index_path, kb_path):
    """Load the FAISS index and knowledge base."""
    index = faiss.read_index(index_path)
    df = pd.read_csv(kb_path)
    return index, df

def retrieve(query, index, df, k=20):
    """Retrieve top-k songs for a query string."""
    # Embed the query
    response = client.embeddings.create(
        input=[query], model="text-embedding-3-small"
    )
    query_vec = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(query_vec)

    # Search
    scores, indices = index.search(query_vec, k)

    # Return results
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results
from openai import OpenAI
import json

client = OpenAI()

def generate_playlist(profile, retrieved_songs_df, prompt_template):
    songs_text = ""
    for _, row in retrieved_songs_df.iterrows():
       

In [5]:
# Test to see if imported files are working
from src.profiling import generate_queries
from src.retrieval import load_retrieval_system, retrieve
from src.generation import generate_playlist
from configs.prompts import GENERATION_PROMPT


print('All imports succeed!')

All imports succeed!


In [8]:
faiss_index, bm25, df = load_retrieval_system(
    "data/index/songs.index",
    "data/index/bm25.pkl",
    "data/processed/knowledge_base.csv"
)

patient = {
    "name": "James Wilson",
    "gender": "Male",
    "birth_year": 1948,
    "hometown": "Detroit, Michigan",
    "cultural_background": "African American",
    "life_events": [
        {"year": 1966, "event": "Graduated from Cass Tech High School"},
        {"year": 1968, "event": "Drafted into the Vietnam War"},
        {"year": 1971, "event": "Married Dorothy in Detroit"},
        {"year": 1975, "event": "First child born"},
    ]
}

retrieved, queries = profile_to_context(patient, faiss_index, bm25, df)
print(f"Retrieved {len(retrieved)} unique songs")

result = generate_playlist(patient, retrieved, GENERATION_PROMPT)

print("\n=== PLAYLIST ===")
for song in result["playlist"]:
    print(f"  {song['rank']}. {song['song']} — {song['artist']} ({song['year']})")
    print(f"     Reason: {song['relevance']}\n")

print("\n=== CAREGIVER CARDS ===")
for card in result["caregiver_cards"]:
    print(f"  Song: {card['song']}")
    print(f"  Prompt: {card['prompt']}\n")

TypeError: load_retrieval_system() takes 2 positional arguments but 3 were given